In [ ]:
%load_ext autoreload
%autoreload 2

from itertools import product
import sys
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D
from PixelGen.metrics import MultiModalVIMetrics
from sklearn.preprocessing import PowerTransformer
from pathlib import Path


import anndata as ad
import pixelator
import torch
import scvi
import scipy
# from scvi import autotune

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

# import ray
# from ray import tune


from PixelGen.pxl_utils import train_model, get_model_latents
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA
from pixelator.common.statistics import clr_transformation, dsb_normalize


from pixelator.pna.plot import molecule_rank_plot
# from pixelator.plot import molecule_rank_plot, cell_count_plot, scatter_umi_per_upia_vs_tau
# from pixelator.statistics import c
# lr_transformation
# from pixelator.analysis.normalization import dsb_normalize


from sklearn.preprocessing import StandardScaler, MinMaxScaler 

from PixelGen.pxl_utils import train_model, get_model_latents, convert_polarization_to_feature_matrix, \
     convert_colocalization_to_feature_matrix, download_pxl
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA, add_one_hot_encoding_obsm, plot_cumulative_variance
from PixelGen.common_utils import standardize, std_clip, filter_hv, split_pair_column, filter_df_by_two_columns, rank_plot
from PixelGen.metrics import MultiModalVIMetrics, distr_autocorrelation_in_latent
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D

import tempfile

from scvi import REGISTRY_KEYS
from scvi.module.base import (
    BaseModuleClass,
    LossOutput,
    PyroBaseModuleClass,
    auto_move_data,
)
from torch.distributions import NegativeBinomial, Normal, Poisson, MixtureSameFamily, Beta
from torch.distributions import kl_divergence as kl



# from cytovi import CytoVI

print(torch.cuda.is_available())
from sklearn.decomposition import PCA

from anndata import AnnData
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)

sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

from utils import plot_latent, plot_gene_heatmap, plot_model_latents,get_dense,calculate_metrics,plot_composite_ppc
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
MODEL_PATH='/home/projects/nyosef/zvise/PixelGen/PixelGen/PBMSC/models/'


In [ ]:
adata=sc.read_h5ad('/home/projects/nyosef/zvise/PixelGen/PixelGen/PBMSC/cache/adata_for_model.h5ad')

GRAPHS_DIR = "/home/projects/nyosef/zvise/PixelGen/PixelGen/PBMSC/GVAE/GRAPHS"
GVAE_NODE_MODEL='/home/projects/nyosef/zvise/PixelGen/PixelGen/PBMSC/GVAE/models/gvae_gat_model.pth'


# CREATE GRAPHS

In [ ]:
DATA_PATH='/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/new_pbmsc'
DATA_DIR = Path(DATA_PATH)
files = [f for f in DATA_DIR.rglob('*.pxl') if f.is_file()]
files

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import gc
import time
from torch_geometric.data import Data
# Using read_pna as you requested
from pixelator import read_pna as read
from joblib import Parallel, delayed
from tqdm import tqdm

# --- 1. SETUP & CONFIGURATION ---
# Updated output directory based on your code snippet
output_dir = "/home/projects/nyosef/zvise/PixelGen/PixelGen/PBMSC/GVAE/GRAPHS"
os.makedirs(output_dir, exist_ok=True)

# Batch size to prevent "Query interrupted" / Memory errors
BATCH_SIZE = 50 

# Your specific file list extracted from previous logs
FILES = files

# --- 2. DEFINE GLOBAL VOCABULARY ---
print("🔵 [INIT] Initializing marker vocabulary from the first file...")
try:
    ref_ds = read(FILES[0])
    ALL_MARKERS = ref_ds.adata().var_names.tolist()
    MARKER_TO_IDX = {m: i for i, m in enumerate(ALL_MARKERS)}
    NUM_CLASSES = len(ALL_MARKERS)
    print(f"✅ [INIT] Vocabulary Fixed: {NUM_CLASSES} markers.")
except Exception as e:
    print(f"❌ [CRITICAL] Failed to initialize vocabulary: {e}")
    raise e

# --- 3. CONVERTER FUNCTION ---
def df_to_pyg_data(edge_df: pd.DataFrame) -> Data:
    if edge_df.empty: return None

    # 1. Map UMI strings to Integer IDs
    all_umis = pd.concat([edge_df["umi1"], edge_df["umi2"]])
    node_codes, unique_umis = pd.factorize(all_umis)

    # 2. Create Edge Index
    src = node_codes[:len(edge_df)]
    dst = node_codes[len(edge_df):]
    edge_index = torch.tensor([src, dst], dtype=torch.long)

    # 3. Create Node Features
    node_lookup = pd.concat([
        edge_df[["umi1", "marker_1"]].rename(columns={"umi1": "umi", "marker_1": "marker"}),
        edge_df[["umi2", "marker_2"]].rename(columns={"umi2": "umi", "marker_2": "marker"})
    ]).drop_duplicates(subset="umi").set_index("umi")

    ordered_markers = node_lookup.reindex(unique_umis)["marker"]
    x_indices = ordered_markers.map(MARKER_TO_IDX).fillna(0).values.astype(int)

    x = torch.nn.functional.one_hot(
        torch.from_numpy(x_indices).long(), 
        num_classes=NUM_CLASSES
    ).float()

    return Data(x=x, edge_index=edge_index)

# --- 4. WORKER FUNCTION ---
def process_single_cell(cell_name, cell_df):
    save_path = os.path.join(output_dir, f"{cell_name}.pt")
    if os.path.exists(save_path): return "skipped"
    
    try:
        pyg_data = df_to_pyg_data(cell_df)
        if pyg_data is not None:
            torch.save(pyg_data, save_path)
            return "saved"
    except: 
        return "error"
    return "empty"

# --- 5. MAIN EXECUTION LOOP (BATCHED) ---
print(f"🔵 [START] Processing {len(FILES)} files into: {output_dir}")

for file_idx, f_path in enumerate(FILES):
    f_name = os.path.basename(f_path)
    print(f"\n📂 [FILE {file_idx+1}/{len(FILES)}] Opening: {f_name}")
    
    try:
        # 1. Open File
        data = read(f_path)
        
        # 2. Get Metadata (Fast)
        all_cells = data.adata().obs_names.tolist()
        total_cells = len(all_cells)
        
        # 3. Process in Batches (Prevents "Query Interrupted")
        pbar = tqdm(range(0, total_cells, BATCH_SIZE), desc=f"   🚀 Processing", unit="batch")
        
        for i in pbar:
            batch_cells = all_cells[i : i + BATCH_SIZE]
            
            try:
                # Load ONLY this batch from DB
                batch_df = data.filter(components=batch_cells).edgelist().to_df()
                
                if batch_df.empty: continue
                
                # Standardize columns
                rename_map = {'upia': 'umi1', 'upib': 'umi2', 'marker': 'marker_1'}
                batch_df = batch_df.rename(columns={k:v for k,v in rename_map.items() if k in batch_df.columns})
                
                # Group & Parallel Process
                grouped = batch_df.groupby("component")
                results = Parallel(n_jobs=-1)(
                    delayed(process_single_cell)(name, group) 
                    for name, group in grouped
                )
                
                # Update stats & Cleanup
                saved = results.count("saved")
                del batch_df, grouped, results
                gc.collect()
                
                pbar.set_postfix_str(f"Saved: {saved}")

            except Exception as e:
                print(f"   ⚠️ Batch Error: {e}")
                continue

    except Exception as e:
        print(f"❌ [FILE ERROR] Could not process {f_name}: {e}")

print("\n🎉 [FINISH] All graphs saved.")

# RUN MODEL

In [ ]:
import torch
import torch.nn as nn
import glob
import random
import numpy as np
import gc
from torch_geometric.nn import VGAE, GATv2Conv
from torch.utils.checkpoint import checkpoint 
import os

# --- 1. CONFIGURATION ---
INPUT_DIM = 158       
HIDDEN_DIM = 64       
LATENT_DIM = 16       
HEADS = 4             # High expressivity
LEARNING_RATE = 0.005 
EPOCHS = 50           
PATIENCE = 10         
MAX_EDGES = 150000    # Skip huge artifacts
VALIDATION_SPLIT = 0.15 # 15% for validation

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Directories
output_dir = GRAPHS_DIR
SAVE_PATH = GVAE_NODE_MODEL

print(f"Using device: {DEVICE} (Mixed Precision + Checkpointing)")

# --- 2. MODEL DEFINITION ---
class VariationalGATEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=HEADS, concat=True)
        hidden_out = hidden_channels * HEADS
        self.conv_mu = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)
        self.conv_logstd = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)

    def run_conv1(self, x, edge_index):
        return self.conv1(x, edge_index).relu()

    def forward(self, x, edge_index):
        # Memory Optimization: Checkpointing
        if self.training: 
            x.requires_grad_(True) 
            x = checkpoint(self.run_conv1, x, edge_index, use_reentrant=False)
        else:
            x = self.run_conv1(x, edge_index)
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

# Initialize
encoder = VariationalGATEncoder(INPUT_DIM, HIDDEN_DIM, LATENT_DIM)
model = VGAE(encoder).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler() 

# --- 3. DATA PREPARATION ---
files = glob.glob(f"{output_dir}/*.pt")
random.shuffle(files)

split_idx = int(len(files) * (1 - VALIDATION_SPLIT))
train_files = files[:split_idx]
val_files = files[split_idx:]

print(f"Total Graphs: {len(files)}")
print(f"   Training: {len(train_files)} (85%)")
print(f"   Validation: {len(val_files)} (15%)")

# --- 4. TRAINING LOOP ---
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    random.shuffle(train_files)
    
    # Clean memory before starting epoch
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    
    count = 0
    for i, file_path in enumerate(train_files):
        try:
            # Load
            data = torch.load(file_path, weights_only=False)
            
            # Check size
            if data.num_edges > MAX_EDGES:
                # Verbose skip
                # print(f"   ⚠️ Skipping huge graph: {os.path.basename(file_path)} ({data.num_edges} edges)")
                del data
                continue

            data = data.to(DEVICE)
            optimizer.zero_grad()
            
            # Mixed Precision Forward
            with torch.cuda.amp.autocast():
                z = model.encode(data.x, data.edge_index)
                loss = model.recon_loss(z, data.edge_index) + (1 / data.num_nodes) * model.kl_loss()
            
            # Backward
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            total_loss += loss.item()
            count += 1
            
            # Verbose progress every 50 files
            if i % 50 == 0:
                print(f"   Batch {i}/{len(train_files)} | Loss: {loss.item():.4f}")
            
            del data, z, loss
            
        except RuntimeError as e:
            if 'out of memory' in str(e):
                print(f"   ❌ OOM Error on file {i}. Skipping.")
                torch.cuda.empty_cache()
                continue
            else:
                raise e

    avg_train_loss = total_loss / count if count > 0 else 0

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss_accum = 0
    val_count = 0
    
    with torch.no_grad():
        for file_path in val_files:
            try:
                data = torch.load(file_path, weights_only=False)
                if data.num_edges > MAX_EDGES: 
                    del data
                    continue
                data = data.to(DEVICE)
                
                z = model.encode(data.x, data.edge_index)
                loss = model.recon_loss(z, data.edge_index) + (1 / data.num_nodes) * model.kl_loss()
                
                val_loss_accum += loss.item()
                val_count += 1
                del data, z, loss
            except:
                torch.cuda.empty_cache()
                continue

    avg_val_loss = val_loss_accum / val_count if val_count > 0 else float('inf')
    
    print(f"✅ Epoch {epoch+1} Summary: Train Loss={avg_train_loss:.4f} | Val Loss={avg_val_loss:.4f}")

    # --- EARLY STOPPING ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"   💾 Best model saved (Val Loss: {best_val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"   ⏳ No improvement. Patience: {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print("🛑 Early stopping triggered.")
            break

print("Training finished.")

In [ ]:
adata.obs.condition.value_counts()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GlobalAttention, GATv2Conv, VGAE
from torch_geometric.utils import softmax
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm
import gc

# --- 1. CONFIGURATION ---
BATCH_SIZE = 8          # Low batch size to prevent OOM
INPUT_DIM = 16          # Latent dim from GVAE
HIDDEN_DIM = 64         # Classifier hidden dim
NUM_HEADS = 4           # Number of biological patterns to learn
EPOCHS = 50
LR = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Paths (Adjust if needed)
GVAE_PATH = GVAE_NODE_MODEL
CACHE_DIR = '/home/projects/nyosef/zvise/PixelGen/PixelGen/PBMSC/GVAE/models/supervised_attention'
OUTPUT_DIR = GRAPHS_DIR # Where your .pt files are
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"🚀 Starting Supervised Attention Pipeline on {DEVICE}")

# --- 2. MODEL DEFINITIONS ---

# A. The Micro-Model (Frozen GVAE) - Must match training exactly
class VariationalGATEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        # Heads=4, concat=True -> Output size is hidden_channels * 4
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=4, concat=True)
        hidden_out = hidden_channels * 4
        # Heads=1, concat=False -> Output size is out_channels
        self.conv_mu = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)
        self.conv_logstd = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

# B. The Macro-Model (Multi-Head Classifier)
class MultiHeadAttentionClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        
        # 1. Independent Attention Gates
        self.attention_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, 32),
                nn.Tanh(),
                nn.Linear(32, 1)
            ) for _ in range(num_heads)
        ])
        
        # 2. Pooling Layers
        self.poolers = nn.ModuleList([
            GlobalAttention(gate_nn=gate) for gate in self.attention_heads
        ])
        
        # 3. Classifier
        # Input is (input_dim * num_heads) because we concat the results
        self.classifier = nn.Sequential(
            nn.Linear(input_dim * num_heads, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x, batch):
        pooled_outputs = []
        for pooler in self.poolers:
            h = pooler(x, batch) 
            pooled_outputs.append(h)
            
        h_concat = torch.cat(pooled_outputs, dim=1)
        logits = self.classifier(h_concat)
        return logits, h_concat

    def get_attention_weights(self, x, batch):
        """Returns list of attention weights (one per head)"""
        weights = []
        for gate in self.attention_heads:
            x_gate = gate(x) 
            alpha = softmax(x_gate, batch)
            weights.append(alpha.detach().cpu().numpy())
        return weights

# --- 3. DATASET & LOADING ---
class LabeledGraphDataset(Dataset):
    def __init__(self, file_list, labels, gvae_model, device):
        super().__init__()
        self.file_list = file_list
        self.labels = labels
        self.gvae_model = gvae_model
        self.device = device
        
        # Freeze GVAE to save memory
        self.gvae_model.eval()
        for param in self.gvae_model.parameters():
            param.requires_grad = False

    def len(self):
        return len(self.file_list)

    def get(self, idx):
        path = self.file_list[idx]
        data = torch.load(path, weights_only=False).to(self.device)
        
        # Transform Raw Graph -> Embeddings
        with torch.no_grad():
            z_nodes = self.gvae_model.encode(data.x, data.edge_index)
        
        # Return embeddings, label, and raw protein IDs
        return Data(
            x=z_nodes.cpu(), 
            y=torch.tensor(self.labels[idx], dtype=torch.long),
            protein_id=data.x.argmax(dim=1).cpu(),
            num_nodes=data.num_nodes
        )

# --- 4. EXECUTION PIPELINE ---

# A. Load Frozen GVAE
print("❄️ Loading Frozen GVAE...")
gvae_encoder = VariationalGATEncoder(158, 64, 16) 
gvae_model = VGAE(gvae_encoder).to(DEVICE)
gvae_model.load_state_dict(torch.load(GVAE_PATH, map_location=DEVICE, weights_only=True))
gvae_model.eval()

# B. Prepare Labels (Control / 4hrs / 24hrs)
print("🏷️ Preparing Data...")
files = glob.glob(f"{OUTPUT_DIR}/*.pt")
valid_files = []
valid_labels = []

# Ensure 'adata' is available in your environment!
id_to_cond = adata.obs['condition'].to_dict() 

for f in files:
    cell_id = os.path.basename(f).replace('.pt', '')
    if cell_id in id_to_cond:
        valid_files.append(f)
        valid_labels.append(id_to_cond[cell_id])

le = LabelEncoder()
y_encoded = le.fit_transform(valid_labels)
classes = le.classes_
print(f"   • Found {len(valid_files)} cells")
print(f"   • Conditions: {dict(zip(classes, range(len(classes))))}")

X_train, X_test, y_train, y_test = train_test_split(
    valid_files, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

train_ds = LabeledGraphDataset(X_train, y_train, gvae_model, DEVICE)
test_ds = LabeledGraphDataset(X_test, y_test, gvae_model, DEVICE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# C. Train Classifier
model = MultiHeadAttentionClassifier(INPUT_DIM, HIDDEN_DIM, len(classes), NUM_HEADS).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

print("\n🔄 Training Classifier...")
best_val_acc = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
    
    for batch in loop:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        
        logits, _ = model(batch.x, batch.batch)
        loss = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == batch.y).sum().item()
        total += batch.y.size(0)
        loop.set_postfix(loss=loss.item(), acc=correct/total)
    
    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE)
            logits, _ = model(batch.x, batch.batch)
            preds = logits.argmax(dim=1)
            val_correct += (preds == batch.y).sum().item()
            val_total += batch.y.size(0)
    
    val_acc = val_correct/val_total
    print(f"   Train Acc: {correct/total:.2f} | Val Acc: {val_acc:.2f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(CACHE_DIR, "best_attention_classifier.pth"))

# --- 5. INTERPRETATION & PLOTS ---

def analyze_attention(model, loader, marker_map, target_class_idx):
    """Finds top proteins driving the decision for a specific class"""
    model.eval()
    protein_scores = {p: [] for p in marker_map.keys()}
    idx_to_marker = {v: k for k, v in marker_map.items()}
    
    print(f"\n🕵️‍♀️ Analyzing Attention for Class Index {target_class_idx}...")
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Scanning"):
            batch = batch.to(DEVICE)
            logits, _ = model(batch.x, batch.batch)
            preds = logits.argmax(dim=1)
            
            # Mask for correctly predicted graphs of target class
            target_mask = (preds == target_class_idx) & (batch.y == target_class_idx)
            if not target_mask.any(): continue
                
            weights_list = model.get_attention_weights(batch.x, batch.batch)
            avg_weights = np.mean(np.array(weights_list), axis=0).flatten()
            
            batch_np = batch.batch.cpu().numpy()
            target_indices = np.where(target_mask.cpu().numpy())[0]
            node_mask = np.isin(batch_np, target_indices)
            
            relevant_weights = avg_weights[node_mask]
            relevant_ids = batch.protein_id.cpu().numpy()[node_mask]
            
            for pid, w in zip(relevant_ids, relevant_weights):
                p_name = idx_to_marker.get(pid, "Unknown")
                protein_scores[p_name].append(w)
    
    results = []
    for p, scores in protein_scores.items():
        if len(scores) > 0:
            results.append({
                "Protein": p, 
                "Mean_Att": np.mean(scores),
                "Max_Att": np.max(scores), 
                "Count": len(scores)
            })
    return pd.DataFrame(results).sort_values("Mean_Att", ascending=False)

# Load best model
model.load_state_dict(torch.load(os.path.join(CACHE_DIR, "best_attention_classifier.pth")))

# Plot Confusion Matrix
print("\n📊 Generating Confusion Matrix...")
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        logits, _ = model(batch.x, batch.batch)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Prediction Accuracy by Condition')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# Analyze Top Proteins for "24h" (Assuming index 1, change based on print output!)


In [ ]:
gf